# 03 — SQL Validation and Reflection

Interactive test bench for the SQL Executor and Reflector nodes.

**What this notebook is for:**
- Load a real `*_kpis.json` (Domain/KPI node output) + its `DataProfile`
- Run the SQL Executor for real, against real data, and inspect what came back
- Assemble the exact reflection prompt `run_reflector` sends
- Call the Reflector LLM live and inspect what it flags
- Confirm the one-retry cycle actually fixes (or correctly drops) a flagged KPI
- Iterate on the reflection prompt without touching `agent/reflector.py` until you're happy

**What this notebook is *not* for:** it doesn't replace `run_sql_executor` / `run_reflector` in the pipeline. Port a validated prompt change back into `agent/reflector.py`'s `SYSTEM_PROMPT` once you're satisfied.


In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from agent.models import DataProfile, KPISpec, QueryResult, ReflectionNote, ReflectionResponse
from agent.state import AgentState
from agent.sql_executor import run_sql_executor
from agent.reflector import (
    run_reflector,
    _build_reflection_prompt,
    _call_reflection_llm,
    SYSTEM_PROMPT,
)

os.environ["LLM_PROVIDER"] = os.environ.get("LLM_PROVIDER", "gemini")  # or "mistral"
print("Testing provider:", os.environ["LLM_PROVIDER"])


## 1. Load a real dataset + its KPIs

Point these at any dataset you've already run through the Domain/KPI node. Needs:
- The original data file (CSV, or the `.sqlite` file for chinook)
- The matching `*_profile.json`
- The matching `*_kpis.json`


In [ ]:
DATASET_NAME = "online_retail"
DATA_PATH = "../data/raw/online_retail.csv"          # the real CSV — get this from your machine
PROFILE_PATH = f"../data/profiles/{DATASET_NAME}_profile.json"
KPIS_PATH = f"../data/profiles/{DATASET_NAME}_kpis.json"

with open(PROFILE_PATH) as f:
    profile = DataProfile(**json.load(f))

with open(KPIS_PATH) as f:
    kpi_data = json.load(f)
kpi_specs = [KPISpec(**k) for k in kpi_data["kpi_specs"]]

print(f"Loaded profile ({profile.row_count} rows) and {len(kpi_specs)} KPIs for '{DATASET_NAME}'")
for k in kpi_specs:
    print(f"  - {k.name} ({k.chart_type})")


## 2. Run the SQL Executor for real

This actually loads `DATA_PATH` into an in-memory SQLite table and runs every KPI's SQL against it.
If `DATA_PATH` doesn't exist locally yet, this cell will raise `FileNotFoundError` — get the real CSV
(or point at the real `.sqlite` file for chinook) before continuing.


In [ ]:
state = AgentState(input_path=DATA_PATH, data_profile=profile, kpi_specs=kpi_specs)
state = run_sql_executor(state)

for qr in state.query_results:
    print(f"\n[{qr.status.upper()}] {qr.kpi_name}")
    print(f"    columns: {qr.columns}, row_count: {qr.row_count}")
    if qr.status == "fail":
        print(f"    error: {qr.error_message}")
    else:
        print(f"    sample: {qr.result_rows[:3]}")

n_failed = sum(1 for qr in state.query_results if qr.status == "fail")
print(f"\n{'ALL QUERIES EXECUTED SUCCESSFULLY' if n_failed == 0 else f'{n_failed} QUERIES FAILED'}")


## 3. Assemble the reflection prompt (edit `SYSTEM_PROMPT` in `agent/reflector.py` to iterate)

This is a live look at exactly what gets sent to the LLM for review — the DataProfile, every
KPI's SQL + reasoning, and its execution result (or error).


In [ ]:
prompt = _build_reflection_prompt(profile, kpi_specs, state.query_results)
print(prompt)


## 4. Call the Reflector LLM live

Uses the real system prompt from `agent/reflector.py`. If you want to test prompt changes,
edit `SYSTEM_PROMPT` in that file directly, restart the kernel (or re-import), and re-run this cell.


In [ ]:
provider = os.environ["LLM_PROVIDER"]
response = _call_reflection_llm(SYSTEM_PROMPT, prompt, provider)

print(f"--- {len(response.notes)} issue(s) found by {provider} ---")
for note in response.notes:
    print(f"\n[{note.issue_type}] {note.kpi_name}")
    print(f"  description: {note.description}")
    print(f"  fix_instruction: {note.fix_instruction}")
    print(f"  fixed_sql_query: {note.fixed_sql_query}")

if not response.notes:
    print("No issues found — every KPI passed review as-is.")


## 5. Judgment call: is the Reflector finding *real* issues?

Read through Section 4's output and ask, for each flagged KPI:
- Is this a genuine problem, or is the model being overly critical of stylistically-different-but-correct SQL?
- If `fixed_sql_query` is provided, does it actually look correct? (Sanity-check it manually before trusting the retry.)
- If nothing was flagged, is that because the KPIs are actually all fine, or because the prompt isn't probing hard enough?
  Try deliberately breaking one KPI's SQL below (Section 6) to check the Reflector's sql_error detection works at all.


In [ ]:
# Scratch cell — log your judgment here as you iterate.
# e.g.: "Retail: flagged the AOV KPI as implausible (correctly — it wasn't filtering
# negative quantities). Fix looks right. HR: flagged nothing, seems genuinely clean."

notes_on_reflector_quality = []


## 6. Sanity check: does the Reflector catch an obviously broken KPI?

Deliberately corrupt one KPI's SQL to a guaranteed failure, re-run the executor and the
reflector, and confirm `sql_error` gets flagged and (if you supply your own believable fix
scenario) the retry path in `run_reflector` behaves as expected.


In [ ]:
import copy

sabotaged_kpis = copy.deepcopy(kpi_specs)
sabotaged_kpis[0].sql_query = "SELECT SUM(ThisColumnDoesNotExist) FROM data"

sabotaged_state = AgentState(input_path=DATA_PATH, data_profile=profile, kpi_specs=sabotaged_kpis)
sabotaged_state = run_sql_executor(sabotaged_state)

print("Executor result for sabotaged KPI:", sabotaged_state.query_results[0].status, sabotaged_state.query_results[0].error_message)

sabotage_prompt = _build_reflection_prompt(profile, sabotaged_kpis, sabotaged_state.query_results)
sabotage_response = _call_reflection_llm(SYSTEM_PROMPT, sabotage_prompt, provider)

flagged_names = [n.kpi_name for n in sabotage_response.notes]
print("Flagged KPIs:", flagged_names)
assert sabotaged_kpis[0].name in flagged_names, "Reflector did not catch the deliberately broken KPI!"
print("OK: Reflector correctly caught the sql_error.")


## 7. Full node test: run_reflector end to end (real LLM, real retry)

This calls the actual `run_reflector` function — not a mock — so it will perform a real
retry cycle if anything gets flagged. Safe to run repeatedly; each call is independent.


In [ ]:
final_state = run_reflector(state, dataset_name=DATASET_NAME)

print(f"KPIs remaining after reflection: {[k.name for k in final_state.kpi_specs]}")
print(f"Reflection notes recorded: {len(final_state.reflection_notes)}")
for qr in final_state.query_results:
    print(f"  {qr.kpi_name}: {qr.status}")

all_pass = all(qr.status == "pass" for qr in final_state.query_results)
print(f"\n{'ALL REMAINING KPIs PASS' if all_pass else 'SOME REMAINING KPIs STILL FAILING — investigate'}")


## 8. Repeat for the other datasets

Change `DATASET_NAME` / `DATA_PATH` in Section 1 to `chinook` / `hr` (with their real data files)
and re-run top to bottom. Per the issue's acceptance criteria, the Reflector should identify at
least one real issue across the three datasets — if none of the three surface anything by the
end of this notebook, that's worth treating as a signal the reflection prompt may need to probe
harder (Section 5's judgment call), not necessarily that everything is perfect.
